# Gold Layer Views

## Executive Overview page

In [0]:
%sql
-- Month-by-month revenue, expenses, net cash flow, and month-over-month change
CREATE OR REPLACE VIEW cash_flow_project.cash_flow_gold.vw_monthly_financial_overview AS
SELECT
    month_key,
    calendar_year,
    quarter,
    ROUND(cash_revenue, 2)      AS revenue,
    ROUND(total_expenses, 2)    AS total_expenses,
    ROUND(net_cash_flow, 2)     AS net_cash_flow,
    ROUND(month_end_balance, 2) AS closing_balance,
    ROUND(
        (cash_revenue - LAG(cash_revenue) OVER (ORDER BY month_key))
        / NULLIF(LAG(cash_revenue) OVER (ORDER BY month_key), 0) * 100, 1
    ) AS revenue_mom_change_pct,
    ROUND(
        (total_expenses - LAG(total_expenses) OVER (ORDER BY month_key))
        / NULLIF(LAG(total_expenses) OVER (ORDER BY month_key), 0) * 100, 1
    ) AS expenses_mom_change_pct
FROM cash_flow_project.cash_flow_gold.fact_monthly_summary
ORDER BY month_key

In [0]:
%sql
-- Which quarter makes the most / least money, across both years
CREATE OR REPLACE VIEW cash_flow_project.cash_flow_gold.vw_quarterly_performance AS
SELECT
    calendar_year,
    quarter,
    CONCAT('Q', quarter, ' ', calendar_year) AS quarter_label,
    ROUND(SUM(cash_revenue), 2)    AS total_revenue,
    ROUND(SUM(total_expenses), 2)  AS total_expenses,
    ROUND(SUM(net_cash_flow), 2)   AS total_net_cash_flow,
    ROUND(AVG(cash_revenue), 2)    AS avg_monthly_revenue,
    RANK() OVER (ORDER BY SUM(cash_revenue) DESC) AS revenue_rank
FROM cash_flow_project.cash_flow_gold.fact_monthly_summary
GROUP BY calendar_year, quarter
ORDER BY calendar_year, quarter

In [0]:
%sql
-- 2022 vs 2023 side by side - is the business actually growing?
CREATE OR REPLACE VIEW cash_flow_project.cash_flow_gold.vw_yoy_comparison AS
SELECT
    calendar_year,
    ROUND(SUM(cash_revenue), 2)          AS total_revenue,
    ROUND(SUM(total_expenses), 2)        AS total_expenses,
    ROUND(SUM(net_cash_flow), 2)         AS total_net_cash_flow,
    ROUND(AVG(payroll_pct_of_revenue), 1) AS avg_payroll_pct_of_revenue,
    ROUND(
        (SUM(cash_revenue) - LAG(SUM(cash_revenue)) OVER (ORDER BY calendar_year))
        / NULLIF(LAG(SUM(cash_revenue)) OVER (ORDER BY calendar_year), 0) * 100, 1
    ) AS revenue_growth_pct_vs_prior_year
FROM cash_flow_project.cash_flow_gold.fact_monthly_summary
GROUP BY calendar_year
ORDER BY calendar_year

In [0]:
%sql
-- "How many months could I survive if revenue stopped today" - a classic owner question
CREATE OR REPLACE VIEW cash_flow_project.cash_flow_gold.vw_cash_runway AS
SELECT
    month_key,
    ROUND(month_end_balance, 2) AS closing_balance,
    ROUND(avg_3m_expenses, 2)   AS avg_3m_expenses,
    ROUND(
        CASE WHEN avg_3m_expenses > 0
             THEN month_end_balance / avg_3m_expenses
             ELSE NULL END, 1
    ) AS runway_months
FROM cash_flow_project.cash_flow_gold.fact_monthly_summary
ORDER BY month_key

## Expense Deep-Dive page

In [0]:
%sql
-- Monthly spend by category, plus each category's share of total expenses
CREATE OR REPLACE VIEW cash_flow_project.cash_flow_gold.vw_expense_breakdown AS
SELECT
    month_key,
    ROUND(cogs_expense, 2)                              AS cogs,
    ROUND(operating_expense, 2)                         AS operating_expense,
    ROUND(total_payroll, 2)                             AS payroll,
    ROUND(total_cc_spend, 2)                            AS credit_card_spend,
    ROUND(total_expenses, 2)                            AS total_expenses,
    ROUND(total_payroll   / NULLIF(total_expenses,0) * 100, 1) AS payroll_pct,
    ROUND(cogs_expense    / NULLIF(total_expenses,0) * 100, 1) AS cogs_pct,
    ROUND(total_cc_spend  / NULLIF(total_expenses,0) * 100, 1) AS credit_card_pct,
    ROUND(operating_expense / NULLIF(total_expenses,0) * 100, 1) AS operating_pct
FROM cash_flow_project.cash_flow_gold.fact_monthly_summary
ORDER BY month_key

In [0]:
%sql
-- Credit card spend by vendor - which vendor is costing the most over time
CREATE OR REPLACE VIEW cash_flow_project.cash_flow_gold.vw_credit_card_vendor_spend AS
SELECT
    date_format(date, 'yyyy-MM') AS month_key,
    description                  AS vendor,
    category_name,
    ROUND(SUM(amount), 2)        AS total_spend,
    COUNT(*)                     AS num_transactions
FROM cash_flow_project.cash_flow_gold.fact_transactions
WHERE account_name = 'Credit Card' AND type = 'Debit' AND category_name != 'Internal Transfer'
GROUP BY date_format(date, 'yyyy-MM'), description, category_name
ORDER BY month_key, total_spend DESC

In [0]:
%sql
-- Payroll by employee/role, with each person's pay as % of that month's revenue
CREATE OR REPLACE VIEW cash_flow_project.cash_flow_gold.vw_payroll_analysis AS
SELECT
    p.month_key,
    p.employee_name,
    p.role,
    p.type AS pay_type,
    p.total_paid,
    m.cash_revenue,
    ROUND(CASE WHEN m.cash_revenue > 0 THEN p.total_paid / m.cash_revenue * 100 ELSE 0 END, 1)
        AS pct_of_monthly_revenue,
    m.shortage_flag
FROM cash_flow_project.cash_flow_gold.fact_payroll p
LEFT JOIN cash_flow_project.cash_flow_gold.fact_monthly_summary m
    ON p.month_key = m.month_key
ORDER BY p.month_key, p.total_paid DESC

## Sales & Products page

In [0]:
%sql
CREATE OR REPLACE VIEW cash_flow_project.cash_flow_gold.vw_sales_product_performance AS
SELECT
    month_key,
    product_category,
    product_type,
    SUM(revenue)              AS total_revenue,
    SUM(transaction_qty)      AS total_units_sold,
    COUNT(transaction_id)     AS num_transactions,
    ROUND(AVG(unit_price), 2) AS avg_unit_price,
    hour,
    CASE
        WHEN hour BETWEEN 6  AND 11 THEN 'Morning (6-11am)'
        WHEN hour BETWEEN 12 AND 14 THEN 'Lunch (12-2pm)'
        WHEN hour BETWEEN 15 AND 17 THEN 'Afternoon (3-5pm)'
        WHEN hour BETWEEN 18 AND 21 THEN 'Evening (6-9pm)'
        ELSE 'Other'
    END AS time_of_day
FROM cash_flow_project.cash_flow_gold.fact_sales
GROUP BY month_key, product_category, product_type, hour
ORDER BY month_key, total_revenue DESC

## AI Recommendations page

In [0]:
%sql
-- Feeds the Groq/Llama AI layer. STARTUP_PERIOD replaces the old false-alarm
-- behavior for a business's first 3 months (see 03_Gold_Layer for why).
CREATE OR REPLACE VIEW cash_flow_project.cash_flow_gold.vw_shortage_detection AS
SELECT
    month_key,
    shortage_flag,
    shortage_severity,
    ROUND(cash_revenue, 2)      AS monthly_income,
    ROUND(total_expenses, 2)    AS monthly_expenses,
    ROUND(net_cash_flow, 2)     AS net_cash_flow,
    ROUND(month_end_balance, 2) AS closing_balance,
    ROUND(avg_3m_revenue, 2)    AS avg_3m_income,
    ROUND(avg_3m_expenses, 2)   AS avg_3m_expenses,
    ROUND(
        CASE WHEN avg_3m_revenue > 0
             THEN (cash_revenue - avg_3m_revenue) / avg_3m_revenue * 100
             ELSE NULL END, 1)  AS income_vs_avg_pct,
    ROUND(
        CASE WHEN avg_3m_expenses > 0
             THEN (total_expenses - avg_3m_expenses) / avg_3m_expenses * 100
             ELSE NULL END, 1)  AS expense_vs_avg_pct,
    needs_ai_suggestion,
    CASE shortage_flag
        WHEN 'CRITICAL'       THEN 'Net cash flow is negative - spent more than earned'
        WHEN 'EXPENSE_SPIKE'  THEN 'Expenses are 20%+ above the 3-month average'
        WHEN 'INCOME_DROP'    THEN 'Revenue dropped 20%+ below the 3-month average'
        WHEN 'WARNING'        THEN 'Balance below 1.5x monthly expenses - low buffer'
        WHEN 'STARTUP_PERIOD' THEN 'Business is in its first 3 months - not enough history for a reliable baseline yet'
        ELSE 'No issues detected'
    END AS flag_explanation
FROM cash_flow_project.cash_flow_gold.fact_monthly_summary
ORDER BY shortage_severity DESC, month_key

## One-stop executive summary
A single row per month with the handful of numbers an owner checks first.
Good default landing view for the Executive Overview page.

In [0]:
%sql
CREATE OR REPLACE VIEW cash_flow_project.cash_flow_gold.vw_business_kpi_summary AS
SELECT
    m.month_key,
    m.quarter,
    ROUND(m.cash_revenue, 2)              AS revenue,
    ROUND(m.total_expenses, 2)            AS expenses,
    ROUND(m.net_cash_flow, 2)             AS net_cash_flow,
    ROUND(m.month_end_balance, 2)         AS closing_balance,
    ROUND(m.payroll_pct_of_revenue, 1)    AS payroll_pct_of_revenue,
    ROUND(CASE WHEN m.avg_3m_expenses > 0
               THEN m.month_end_balance / m.avg_3m_expenses ELSE NULL END, 1) AS cash_runway_months,
    p.pos_gross_revenue,
    m.shortage_flag,
    m.needs_ai_suggestion
FROM cash_flow_project.cash_flow_gold.fact_monthly_summary m
LEFT JOIN (SELECT month_key, SUM(revenue) AS pos_gross_revenue
           FROM cash_flow_project.cash_flow_gold.fact_sales GROUP BY month_key) p
    ON m.month_key = p.month_key
ORDER BY m.month_key